In [6]:
### DICE COEFFICENT ACCURASY

import os
import numpy as np
import pandas as pd
from PIL import Image

# ================== CONFIG (EDIT THESE) ==================
CSV_PATH = "/zpool/vladlab/active_drive/omaltz/scripts/geogaze/coco_decoder/geogaze_model_predictions_individuation.csv"

BASE_MASK_DIR   = "/zpool/vladlab/active_drive/omaltz/scripts/geogaze/decoder_model/stimuli/out/test_stimuli/test_acc_masks"
BASE_PRED_ROOT = "/zpool/vladlab/active_drive/omaltz/scripts/geogaze/coco_decoder/individuation"

THRESH_PRED = 128   # set None to auto
THRESH_T    = 128
THRESH_D    = 128
# ========================================================

# ---- I/O + mask-safe resize ----
def load_gray(path: str) -> np.ndarray:
    """Load an image as grayscale float64 numpy array."""
    return np.asarray(Image.open(path).convert("L"), dtype=np.float64)

def nn_resize(arr: np.ndarray, new_shape) -> np.ndarray:
    """Nearest-neighbor resize for 2D arrays (H,W)."""
    h, w = arr.shape
    H, W = new_shape
    if (H, W) == (h, w):
        return arr
    r = (np.arange(H) * h) // H
    c = (np.arange(W) * w) // W
    return arr[np.ix_(r, c)]

# ---- Binarize with polarity handling (True = object) ----
def binarize(mask: np.ndarray, *, object_is_black: bool, thresh=None) -> np.ndarray:
    """
    Convert grayscale mask into boolean mask where True = object pixel.

    object_is_black:
        True  -> object dark on light background -> use m < thresh
        False -> object light on dark background -> use m > thresh
    """
    m = mask
    if thresh is None:
        thresh = 0.5 * (float(m.min()) + float(m.max()))
    return (m < thresh) if object_is_black else (m > thresh)

# ---- Dice coefficient for boolean masks ----
def dice_from_bool(mask1: np.ndarray, mask2: np.ndarray) -> float:
    """
    Dice coefficient between two boolean masks (True = object pixel).

    Returns 0.0 if both masks are empty.
    """
    m1 = mask1.astype(bool)
    m2 = mask2.astype(bool)

    inter = np.logical_and(m1, m2).sum()
    size1 = m1.sum()
    size2 = m2.sum()

    denom = size1 + size2
    if denom == 0:
        # both empty -> define Dice = 0
        return 0.0

    return 2.0 * inter / denom

# ---- Public API: Dice-based continuous score ----
def allpairs_dice_score(pred_img, target_img, distractor_img,
                        pred_is_white_on_black=True,
                        t_is_black_on_white=True,
                        d_is_black_on_white=True,
                        thresh_pred=None, thresh_t=None, thresh_d=None):
    """
    Compute Dice(pred, target) and Dice(pred, distractor), then form a
    continuous accuracy score in [0,1]:

        acc = Dice(pred, target) / (Dice(pred, target) + Dice(pred, distractor))

    Higher acc = more target-like.
    """

    # 1) Binarize -> True = object pixel
    P_bool = binarize(pred_img,   object_is_black=not pred_is_white_on_black, thresh=thresh_pred)
    T_bool = binarize(target_img, object_is_black=t_is_black_on_white,        thresh=thresh_t)
    D_bool = binarize(distractor_img, object_is_black=d_is_black_on_white,    thresh=thresh_d)

    # 2) Edge case: no prediction pixels -> neutral
    n_pred = P_bool.sum()
    n_targ = T_bool.sum()
    n_dist = D_bool.sum()

    if n_pred == 0:
        return {
            "dice_target": np.nan,
            "dice_distractor": np.nan,
            "accuracy": 0.5,  # neutral between target (1) and distractor (0)
            "counts": {"n_pred": int(n_pred), "n_target": int(n_targ), "n_distractor": int(n_dist)},
            "choice": "NO_PREDICTION",
        }

    # 3) Dice overlaps
    dice_t = dice_from_bool(P_bool, T_bool)
    dice_d = dice_from_bool(P_bool, D_bool)

    # 4) Continuous accuracy via ratio of Dice scores
    denom = dice_t + dice_d
    if denom > 0:
        accuracy = dice_t / denom  # higher = prediction overlaps target more than distractor
    else:
        # no overlap with either -> treat as neutral
        accuracy = 0.5

    # 5) Discrete choice (if you still want it)
    if dice_t > dice_d:
        choice = "TARGET"
    elif dice_d > dice_t:
        choice = "DISTRACTOR"
    else:
        choice = "TIE"

    return {
        "dice_target": dice_t,
        "dice_distractor": dice_d,
        "accuracy": accuracy,
        "counts": {"n_pred": int(n_pred), "n_target": int(n_targ), "n_distractor": int(n_dist)},
        "choice": choice,
    }

# ---- Helper to build full paths for a row ----
def build_paths_for_row(row):
    target_path     = os.path.join(BASE_MASK_DIR, row["target"])
    distractor_path = os.path.join(BASE_MASK_DIR, row["distractor"])

    model_folder = row["model"]  # e.g. cornetIDIV_maskL_bc_bs
    pred_fname   = row["prediction"]  # e.g. test_bc_bs_LR_mask.png

    pred_path = os.path.join(
        BASE_PRED_ROOT,
        model_folder,
        f"{model_folder}_masks",
        pred_fname
    )

    return pred_path, target_path, distractor_path



# ================== RUN OVER CSV ==================
df = pd.read_csv(CSV_PATH)

print(df[["model","prediction"]].head(5))

p, t, d = build_paths_for_row(df.iloc[0])
print("Example pred path:", p)
print("Exists?", os.path.exists(p))


for idx, row in df.iterrows():
    try:
        pred_path, targ_path, dist_path = build_paths_for_row(row)

        # Load
        pred = load_gray(pred_path)
        targ = load_gray(targ_path)
        dist = load_gray(dist_path)

        # Align sizes (match target shape)
        pred = nn_resize(pred, targ.shape)
        dist = nn_resize(dist, targ.shape)

        # Run Dice-based metric
        res = allpairs_dice_score(
            pred_img=pred,
            target_img=targ,
            distractor_img=dist,
            pred_is_white_on_black=True,
            t_is_black_on_white=True,
            d_is_black_on_white=True,
            thresh_pred=THRESH_PRED,
            thresh_t=THRESH_T,
            thresh_d=THRESH_D,
        )

        acc = res["accuracy"]

    except Exception as e:
        print(f"Row {idx}: ERROR ({e})")
        acc = np.nan  # NaN for true errors (missing file, etc.)

    # write continuous accuracy into a new column
    df.at[idx, "acc_continuous"] = acc

# Save out new CSV with continuous accuracy filled
out_path = os.path.splitext(CSV_PATH)[0] + "_cornetIDIV_01_29_26.csv"
df.to_csv(out_path, index=False)
print(f"Done. Saved updated CSV to:\n{out_path}")


                    model              prediction
0  cornetIDIV_maskL_bc_bs  test_bc_bs_LR_mask.png
1  cornetIDIV_maskL_bc_bs  test_bc_bc_LR_mask.png
2  cornetIDIV_maskL_bc_bs  test_bs_bs_LR_mask.png
3  cornetIDIV_maskL_bc_bs  test_bc_bs_UD_mask.png
4  cornetIDIV_maskL_bc_bs  test_bs_bc_UD_mask.png
Example pred path: /zpool/vladlab/active_drive/omaltz/scripts/geogaze/coco_decoder/individuation/cornetIDIV_maskL_bc_bs/cornetIDIV_maskL_bc_bs_masks/test_bc_bs_LR_mask.png
Exists? True
Done. Saved updated CSV to:
/zpool/vladlab/active_drive/omaltz/scripts/geogaze/coco_decoder/geogaze_model_predictions_individuation_cornetIDIV_01_29_26.csv
